# Imports & Directory Setup

In [1]:
import pandas as pd
import numpy as np
from scipy.integrate import quad
import scipy.linalg as la
import matplotlib.pyplot as plt

import emcee
import corner
import os
import time
from multiprocess import Pool, cpu_count

# Create necessary directories
os.makedirs("mcmc_chains", exist_ok=True)
os.makedirs("plots", exist_ok=True)

# Prevent internal SciPy multithreading conflicts
os.environ["OMP_NUM_THREADS"] = "1"
print("Environment initialized successfully.")

Environment initialized successfully.


# Load SNe Dataset & Covariance Matrix

In [2]:
# ---------------------------------------------------------
# Load Pantheon+SH0ES Data and Full Covariance Matrix
# ---------------------------------------------------------
df_sn = pd.read_csv('../../data/Pantheon+SH0ES.dat', sep=r'\s+')
N_sn = len(df_sn)

with open('../../data/Pantheon+SH0ES_STAT+SYS.cov', 'r') as f:
    lines = f.readlines()
if len(lines[0].split()) == 1: 
    lines = lines[1:]
    
C_sn = np.loadtxt(lines).reshape((N_sn, N_sn))
c_chol_sn, lower_sn = la.cho_factor(C_sn)
C_inv_sn = la.cho_solve((c_chol_sn, lower_sn), np.eye(N_sn))
log_det_C_sn = 2.0 * np.sum(np.log(np.diag(c_chol_sn)))

print(f"Pantheon+SH0ES loaded: {N_sn} supernovae.")

Pantheon+SH0ES loaded: 1701 supernovae.


# CPL Physics & Likelihood Engine (SNe Only)

In [3]:
# ---------------------------------------------------------
# CPL Physics & SNe Likelihood Engine (5 Parameters)
# ---------------------------------------------------------
c = 299792.458 # km/s

def E_inverse(z, Omega_m, w0, wa):
    term_matter = Omega_m * (1 + z)**3
    exponent = 3 * (1 + w0 + wa)
    exponential_term = np.exp(-3 * wa * (z / (1 + z)))
    term_de = (1.0 - Omega_m) * ((1 + z)**exponent) * exponential_term
    return 1.0 / np.sqrt(term_matter + term_de)

def comoving_distance_mpc(z, H_0, Omega_m, w0, wa):
    integral, _ = quad(E_inverse, 0, z, args=(Omega_m, w0, wa))
    return (c / H_0) * integral

def luminosity_distance_mpc(z_hel, z_cmb, H_0, Omega_m, w0, wa):
    return (1 + z_hel) * comoving_distance_mpc(z_cmb, H_0, Omega_m, w0, wa)

def model_cosmology_sn(z_hel, z_cmb, M_B, H_0, Omega_m, w0, wa):
    d_L = luminosity_distance_mpc(z_hel, z_cmb, H_0, Omega_m, w0, wa)
    return M_B + 5.0 * np.log10(d_L) + 25.0

def log_prior(params):
    Omega_m, H_0, M_B, w0, wa = params
    
    # Flat physical boundary box
    bounds_check = (
        (0.01 < Omega_m < 0.99) and 
        (50.0 < H_0 < 100.0) and 
        (-21.0 < M_B < -18.0) and 
        (-3.0 < w0 < 1.0) and 
        (-3.0 < wa < 3.0)
    )
    if not bounds_check:
        return -np.inf
        
    # External SH0ES Gaussian Prior for M_B
    lp_MB = -0.5 * ((M_B - (-19.253)) / 0.027)**2
    return lp_MB

def log_posterior_sn(params):
    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf
        
    Omega_m, H_0, M_B, w0, wa = params
    delta = np.zeros(N_sn)
    
    for i, row in df_sn.iterrows():
        m_b_th = model_cosmology_sn(row['zHEL'], row['zCMB'], M_B, H_0, Omega_m, w0, wa)
        delta[i] = row['m_b_corr'] - m_b_th
        
    chi2 = np.dot(delta.T, np.dot(C_inv_sn, delta))
    ll_sn = -0.5 * (chi2 + log_det_C_sn + N_sn * np.log(2 * np.pi))
    
    return lp + ll_sn

# Uniform Random Initialization & MCMC Execution

In [4]:
# ---------------------------------------------------------
# MCMC Execution with Gelman-Rubin Early Stopping
# ---------------------------------------------------------
def compute_gelman_rubin(chain):
    nsteps, nwalkers, ndim = chain.shape
    R_hat = np.zeros(ndim)
    for d in range(ndim):
        samples = chain[:, :, d]
        W = np.mean(np.var(samples, axis=0, ddof=1))
        chain_means = np.mean(samples, axis=0)
        grand_mean = np.mean(chain_means)
        B = (nsteps / (nwalkers - 1.0)) * np.sum((chain_means - grand_mean)**2)
        V_hat = ((nsteps - 1.0) / nsteps) * W + (B / nsteps)
        R_hat[d] = np.sqrt(V_hat / W) if W > 0 else np.inf
    return R_hat

ndim = 5 
nwalkers = 50
max_steps = 20000
check_interval = 200
R_hat_threshold = 1.02 # Safe threshold for degenerate CPL space

# Define uniform prior ranges for each parameter
prior_bounds = [
    (0.01, 0.99),   # Omega_m
    (50.0, 100.0),  # H_0
    (-21.0, -18.0), # M_B
    (-3.0, 1.0),    # w_0
    (-3.0, 3.0)     # w_a
]

# Generate walkers uniformly distributed across the entire prior volume
initial_pos = np.zeros((nwalkers, ndim))
for i in range(ndim):
    low, high = prior_bounds[i]
    initial_pos[:, i] = np.random.uniform(low, high, size=nwalkers)

print("Walkers initialized uniformly across the full prior boundaries:")
for i, name in enumerate(["Omega_m", "H_0", "M_B", "w_0", "w_a"]):
    print(f"  {name}: [{prior_bounds[i][0]}, {prior_bounds[i][1]}]")

# Set up HDFBackend to write to disk
h5_filename = "mcmc_chains/cpl_sne.h5"
backend = emcee.backends.HDFBackend(h5_filename)
backend.reset(nwalkers, ndim)

ncpu = int(max(1, cpu_count() - 2))
print(f"\nStarting SNe-Only CPL MCMC on {ncpu} cores (Max {max_steps} steps)...")

start_time = time.time()
with Pool(processes=ncpu) as pool:
    sampler = emcee.EnsembleSampler(
        nwalkers, ndim, log_posterior_sn, 
        pool=pool, backend=backend
    )
    
    for sample in sampler.sample(initial_pos, iterations=max_steps, progress=True,progress_kwargs={"bar_format": "{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_noinv_fmt}]"}):
        if sampler.iteration % check_interval == 0 and sampler.iteration > 0:
            chain = sampler.get_chain()
            burn_in = int(chain.shape[0] * 0.5)
            R_hat = compute_gelman_rubin(chain[burn_in:, :, :])
            
            if np.all(R_hat < R_hat_threshold):
                print(f"\n✅ Gelman-Rubin convergence achieved at step {sampler.iteration}!")
                print(f"Final R-hat values: {np.round(R_hat, 4)}")
                break

end_time = time.time()
print(f"\nRun complete! Time taken: {(end_time - start_time) / 60:.2f} minutes.")
print(f"Chain saved permanently to: {h5_filename}")

Walkers initialized uniformly across the full prior boundaries:
  Omega_m: [0.01, 0.99]
  H_0: [50.0, 100.0]
  M_B: [-21.0, -18.0]
  w_0: [-3.0, 1.0]
  w_a: [-3.0, 3.0]

Starting SNe-Only CPL MCMC on 18 cores (Max 20000 steps)...


  0%|▎                                                                           | 94/20000 [03:27<12:13:15,  0.45it/s]

emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:emcee: Exception while calling your likelihood function:


emcee: Exception while calling your likelihood function:








  params:
  params:
  params:  params:  params:

KeyboardInterrupt: 

# Read Directly from .h5 File & Plot Diagnostics

In [ ]:
# ---------------------------------------------------------
# Read from .h5 File & Produce Diagnostic Plots
# ---------------------------------------------------------
h5_filename = "mcmc_chains/cpl_sne.h5"

try:
    # Read chain directly from disk
    reader = emcee.backends.HDFBackend(h5_filename)
    full_chain = reader.get_chain()
    
    total_steps, nwalkers, ndim = full_chain.shape
    print(f"Successfully read from {h5_filename}")
    print(f"Chain Dimensions -> Steps: {total_steps}, Walkers: {nwalkers}, Params: {ndim}")
    
    # Automatically discard first 50% as burn-in
    discard_steps = int(total_steps * 0.5)
    thin_steps = 15
    flat_samples = reader.get_chain(discard=discard_steps, thin=thin_steps, flat=True)
    
    labels = [r"$\Omega_m$", r"$H_0$", r"$M_B$", r"$w_0$", r"$w_a$"]
    
    # 1. TRACE PLOT
    fig, axes = plt.subplots(ndim, figsize=(10, 10), sharex=True)
    for i in range(ndim):
        axes[i].plot(full_chain[:, :, i], "k", alpha=0.25)
        axes[i].set_ylabel(labels[i], fontsize=14)
        axes[i].axvline(x=discard_steps, color='red', linestyle='--', label='Burn-in cut')
        if i == 0: axes[i].legend(loc='upper right')
        
    axes[-1].set_xlabel("Step Number", fontsize=12)
    plt.suptitle("Trace Plot: SNe Only (CPL Model from .h5)", y=1.02, fontsize=16)
    plt.tight_layout()
    
    trace_save_path = "plots/cpl_sne_trace.png"
    plt.savefig(trace_save_path, dpi=300, bbox_inches='tight')
    print(f"Trace plot saved to: {trace_save_path}")
    plt.show()

    # 2. CORNER PLOT
    fig = corner.corner(
        flat_samples, 
        labels=labels,
        quantiles=[0.16, 0.5, 0.84], 
        show_titles=True,
        title_kwargs={"fontsize": 12}, 
        color='darkred'
    )
    plt.suptitle("Posterior: SNe Only (CPL Model from .h5)", y=1.02, fontsize=16)
    
    corner_save_path = "plots/cpl_sne_corner.png"
    plt.savefig(corner_save_path, dpi=300, bbox_inches='tight')
    print(f"Corner plot saved to: {corner_save_path}")
    plt.show()
    
except FileNotFoundError:
    print(f"Error: Could not find {h5_filename}. Please run the MCMC cell first!")